<div style="text-align: center">
<img src="https://github.com/LinkedEarth/Logos/blob/master/PyleoTUPS/pyleotups_logo.png?raw=true" alt="PyleoTUPS logo" width="400">
</div>

# Querying the NOAA NCEI Database

## Authors

Deborah Khider [![ORCID](https://img.shields.io/badge/ORCID-0000--0001--7501--8430-A6CE39?logo=orcid)](https://orcid.org/0000-0001-7501-8430), Dhiren Oswal [![ORCID](https://img.shields.io/badge/ORCID-0009--0001--2495--2626-A6CE39?logo=orcid)](https://orcid.org/0009-0001-2495-2626)


## Preamble

In a [previous chapter](02_a_NOAAObject.ipynb), you learned how to create a `NOAADataset` object and its many functionalities. This tutorial demonstrates how to query the database.

A list of search parameters is available on the [NOAA website](https://www.ncei.noaa.gov/access/paleo-search/api) and is summarized below:

* xml_id : str, optional
         Specify the internal XML document ID. Must be an exact match (e.g., '1840').

* noaa_id : str, optional
        Provide the unique NOAA Study ID as a number (e.g., '13156').

* search_text : str, optional
        General text search across study content. Supports wildcards (%) and logical operators (AND, OR).
        Examples: 'younger dryas', 'loess AND stratigraphy'

* data_publisher : by default 'NOAA'
        Choose from: 'NOAA', 'NEOTOMA', or 'PANGAEA'.
        Example: 'NOAA'

* data_type_id : str, optional
        Filter by data type. Use one or more type IDs separated by '|'.
        Available IDs:
                1: BOREHOLE, 2: CLIMATE FORCING, 3: CLIMATE RECONSTRUCTIONS, 4: CORALS AND SCLEROSPONGES,
                6: HISTORICAL, 7: ICE CORES, 8: INSECT, 9: LAKE LEVELS, 10: LOESS,
                11: PALEOCLIMATIC MODELING, 12: FIRE HISTORY, 13: PALEOLIMNOLOGY, 14: PALEOCEANOGRAPHY,
                15: PLANT MACROFOSSILS, 16: POLLEN, 17: SPELEOTHEMS, 18: TREE RING,
                19: OTHER COLLECTIONS, 20: INSTRUMENTAL, 59: SOFTWARE, 60: REPOSITORY
        Example: '4', '4|18'

* investigators : str or list[str], optional
        Investigator(s) in the form ``"LastName, Initials"``. Lists are joined with ``|``.
        
* investigators_and_or : {"and","or"}, default "or"
        Logical combiner when multiple investigators are supplied. Only sent when 2+ items.
        
* locations : str or list[str], optional
        Location(s) as hierarchical strings using ``>`` (e.g., ``"Continent>Africa>Kenya"``). Lists joined with ``|``.
        
* locations_and_or : {"and","or"}, default "or"
        Logical combiner for multiple locations. Only sent when 2+ items.
        
* keywords : str or list[str], optional
        Controlled keyword(s); hierarchies with ``>``. Lists joined with ``|``.
        
* keywords_and_or : {"and","or"}, default "or"
        Logical combiner for multiple keywords. Only sent when 2+ items.
        
* species : str or list[str], optional
        Four-letter tree species codes (uppercase enforced). Lists joined with ``|``.
        
* species_and_or : {"and","or"}, default "or"
        Logical combiner for multiple species. Only sent when 2+ items.
        
* variable_name : str or list[str], optional
        Refers to PaST "cvWhats” terms (hierarchies with ``>``). Lists joined with ``|``.
        
* variable_name_and_or : {"and","or"}, default "or"
        Logical combiner for multiple cvWhats/variable_name. Only sent when 2+ items.
        
* cv_materials : str or list[str], optional
        PaST “Material” terms (hierarchies with ``>``). Lists joined with ``|``.
        
* cv_materials_and_or : {"and","or"}, default "or"
        Logical combiner for multiple cv_materials. Only sent when 2+ items.
        
* cv_seasonalities : str or list[str], optional
        PaST “Seasonality” terms (e.g., ``"annual"`` or ``"3-month>Aug-Oct"``). Lists joined with ``|``.
        
* cv_seasonalities_and_or : {"and","or"}, default "or"
        Logical combiner for multiple cv_seasonalities. Only sent when 2+ items.
        
* min_lat, max_lat : int, optional
        Latitude bounds in whole degrees (–90..90).
        
* min_lon, max_lon : int, optional
        Longitude bounds in whole degrees (–180..180).
        
* min_elevation, max_elevation : int, optional
        Elevation bounds in meters (integers; negative allowed).
        
* earliest_year, latest_year : int, optional
        Year bounds (integers; negative allowed). If provided without time settings, ``time_format`` defaults to ``'CE'``.
        
* time_format : {"CE","BP"}, optional
        Interpretation of years. If omitted with a time window, defaults to ``'CE'``.
        
* time_method : {"overAny","entireOver","overEntire"}, optional
        How to apply the time window (overlap, envelop, or within).
        
* reconstruction : bool or str, optional
        Accepts True/False or strings (case-insensitive) like ``"true"|"yes"|"y"|"1"`` → ``'Y'`` and
        ``"false"|"no"|"n"|"0"`` → ``'N'``. ``None`` omits the filter.
        
* recent : bool, default False
        If True, restrict to studies from the last ~2 years (newest first).
        
* limit : int, default 100
        Number of studies to return (PyleoTUPS default).
        
* display : bool, default False
        If True, render a small preview after parsing. 
         
* skip : int, optional
        Number of studies to skip (for pagination). Use with ``limit`` to page through results.
        Example: ``limit=10, skip=10`` returns items 11–20.

### Goals

- Perform query on NOAA NCEI using their search parameters. 

### Prerequisite

- Understanding of NOAA datasets and associated search API
- [PyleoTUPS NOAADataset object](02_a_NOAAObject.ipynb)

### Reading time

15 min

Let's import our packages!

In [1]:
import pyleotups as pt
import pandas as pd

## Investigator Query

One of the most common search is to look up the datasets produced by an investigator. For instance, let's have a look at some of the studies contributed by one of the authors of these tutorials:

In [2]:
ds = pt.NOAADataset()
res = ds.search_studies(investigators = "Khider, D.")

[2026-04-21 12:45:11,219][INFO] - search_studies: Limit defaulted to 100 (PyleoTUPS).
[2026-04-21 12:45:11,221][INFO] - search_studies: Input Query includes geographical bounds. Inspect the results to ensure they match your intended region as one study can contain sites across various parts of the world.


Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&limit=100&investigators=Khider%2C+D.


Parsing NOAA studies: 100%|██████████| 3/3 [00:00<00:00, 4808.14it/s]
[2026-04-21 12:45:50,085][INFO] - Retrieved 3 studies.


<div style="border-left: 4px solid #1f77b4; background: #f0f8ff; padding: 0.5em 1em; border-radius: 4px;">
  <strong>Note:</strong> NOAA expects Last Name, Initial.
</div>

Let's have a look at the results:

In [3]:
display(res)

,StudyID,XMLID,StudyName,DataType,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,"Coverage [S, N, W, E]",StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding
0,18315,16017,Makassar Strait - Single specimens of P. obliq...,PALEOCEANOGRAPHY,1246,99,704,1851,"(1.4033, 1.4033, 119.078, 119.078)","This dataset contains the d18O, d13C, and weig...","[Medieval Climate Anomaly (MCA), Little Ice Ag...","Deborah Khider, Lowell Stott, Julien Emile-Gea...","[{'Author': 'Khider, D., L. Stott, J. Emile-Ge...","[[{'DataTableID': '28674', 'DataTableName': 'M...",[{'fundingAgency': 'US National Science Founda...
1,8630,2151,"Reuter et al. 2009 Cascayunga Cave, Peru 1000 ...",SPELEOTHEMS,862,-55,1088,2005,"(-6.067, -6.067, -77.183, -77.183)",Oxygen isotope data (d18O) from stalagmites fr...,"[PAGES LOTRED SA2k, PAGES 2k Network]","Hai Cheng, R. Lawrence Edwards, Deborah Khider...","[{'Author': 'Cheng, H.; Edwards, R.L.; Khider,...","[[{'DataTableID': '12492', 'DataTableName': 'C...",[{'fundingAgency': 'US National Science Founda...
2,16055,13818,Western Tropical Pacific SST and Isotope Data ...,PALEOCEANOGRAPHY,11031,199,-9081,1751,"(6.3, 6.3, 125.83, 125.83)",Benthic (Cibicicoides mundulus) foraminifera c...,None,"Deborah Khider, Charles Jackson, Lowell Stott","[{'Author': 'Stott, L.D., K.G. Cannariato, R.C...","[[{'DataTableID': '26023', 'DataTableName': 'M...",[{'fundingAgency': 'US National Science Founda...


What if I am really interested in that last study? Well, one solution is to create a search with the StudyID (they come in handy!) or one can refine the search with another investigator. 

<div style="
    padding: 10px; 
    background-color: #fff3cd; 
    border-left: 6px solid #ffcc00; 
    margin: 10px 0;">
    <strong>Warning:</strong> Performing another search on the same object will erase the previous search results. 
</div>

In [4]:
res = ds.search_studies(investigators = ["Khider, D.", "Jackson, C."])
display(res)

[2026-04-21 12:45:55,641][INFO] - search_studies: Limit defaulted to 100 (PyleoTUPS).
[2026-04-21 12:45:55,641][INFO] - search_studies: Input Query includes geographical bounds. Inspect the results to ensure they match your intended region as one study can contain sites across various parts of the world.


Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&limit=100&investigators=Khider%2C+D.%7CJackson%2C+C.&investigatorsAndOr=or


Parsing NOAA studies: 100%|██████████| 4/4 [00:00<00:00, 6770.47it/s]
[2026-04-21 12:48:03,119][INFO] - Retrieved 4 studies.


,StudyID,XMLID,StudyName,DataType,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,"Coverage [S, N, W, E]",StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding
0,14988,13166,"Espiritu Santo, Vanuatu 446 Year Stalagmite Ox...",SPELEOTHEMS,393,-53,1557,2003,"(-15.5333, -15.5333, 167.0167, 167.0167)",Data include a table of U-Th ages and the d18O...,"[Pacific Decadal Oscillation, hydrology]","Judson Partin, Terrence Quinn, Chuan-Chou Shen...","[{'Author': 'Partin, J.W., T.M. Quinn, C-C She...","[[{'DataTableID': '25015', 'DataTableName': 'T...",[]
1,18315,16017,Makassar Strait - Single specimens of P. obliq...,PALEOCEANOGRAPHY,1246,99,704,1851,"(1.4033, 1.4033, 119.078, 119.078)","This dataset contains the d18O, d13C, and weig...","[Medieval Climate Anomaly (MCA), Little Ice Ag...","Deborah Khider, Lowell Stott, Julien Emile-Gea...","[{'Author': 'Khider, D., L. Stott, J. Emile-Ge...","[[{'DataTableID': '28674', 'DataTableName': 'M...",[{'fundingAgency': 'US National Science Founda...
2,8630,2151,"Reuter et al. 2009 Cascayunga Cave, Peru 1000 ...",SPELEOTHEMS,862,-55,1088,2005,"(-6.067, -6.067, -77.183, -77.183)",Oxygen isotope data (d18O) from stalagmites fr...,"[PAGES LOTRED SA2k, PAGES 2k Network]","Hai Cheng, R. Lawrence Edwards, Deborah Khider...","[{'Author': 'Cheng, H.; Edwards, R.L.; Khider,...","[[{'DataTableID': '12492', 'DataTableName': 'C...",[{'fundingAgency': 'US National Science Founda...
3,16055,13818,Western Tropical Pacific SST and Isotope Data ...,PALEOCEANOGRAPHY,11031,199,-9081,1751,"(6.3, 6.3, 125.83, 125.83)",Benthic (Cibicicoides mundulus) foraminifera c...,None,"Deborah Khider, Charles Jackson, Lowell Stott","[{'Author': 'Stott, L.D., K.G. Cannariato, R.C...","[[{'DataTableID': '26023', 'DataTableName': 'M...",[{'fundingAgency': 'US National Science Founda...


Oops! Not exactly what you were expecting, right? Now, I have 4 studies. Let's have a look at the investigator field: 

In [5]:
for value in res['Investigators']:
    print(value)

Judson Partin, Terrence Quinn, Chuan-Chou Shen, Julien Emile-Geay, Frederick Taylor, Christopher Maupin, Ke Lin, Charles Jackson, Jay Banner, Daniel Sinclair, Chih-An Huh
Deborah Khider, Lowell Stott, Julien Emile-Geay, Robert Thunell, Doug Hammond
Hai Cheng, R. Lawrence Edwards, Deborah Khider, Ashish Sinha, Lowell Stott, Justin Reuter
Deborah Khider, Charles Jackson, Lowell Stott


The reason is that NOAA uses ``or`` by default. So we have been searching for studies whose investigators are either Charles Jackson or Deborah Khider. This is how you perform an ``and`` search:

In [11]:
res = ds.search_studies(investigators = ["Khider, D.", "Jackson, C."], investigators_and_or = 'and')
display(res)

[2026-04-06 16:02:01,180][INFO] - search_studies: Limit defaulted to 100 (PyleoTUPS).
[2026-04-06 16:02:01,181][INFO] - search_studies: Input Query includes geographical bounds. Inspect the results to ensure they match your intended region as one study can contain sites across various parts of the world.


Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&limit=100&investigators=Khider%2C+D.%7CJackson%2C+C.&investigatorsAndOr=and


Parsing NOAA studies: 100%|██████████| 1/1 [00:00<00:00, 1150.70it/s]
[2026-04-06 16:02:32,083][INFO] - Retrieved 1 studies.


,StudyID,XMLID,StudyName,DataType,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,"Coverage [S, N, W, E]",StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding
0,16055,13818,Western Tropical Pacific SST and Isotope Data ...,PALEOCEANOGRAPHY,11031,199,-9081,1751,"(6.3, 6.3, 125.83, 125.83)",Benthic (Cibicicoides mundulus) foraminifera c...,None,"Deborah Khider, Charles Jackson, Lowell Stott","[{'Author': 'Stott, L.D., K.G. Cannariato, R.C...","[[{'DataTableID': '26023', 'DataTableName': 'M...",[{'fundingAgency': 'US National Science Founda...


Which is the study we wanted!

<div style="border-left: 4px solid #1f77b4; background: #f0f8ff; padding: 0.5em 1em; border-radius: 4px;">
  <strong>Note:</strong> Make sure you check which other parameters may have a logical "or" for the searches by default!
</div>


## Geographical Query

One of the most common types of searches is a geographical query. Let's look for all the datasets within 5°S-5°N and 109-125°E, roughly corresponding to the Indo-Pacific Warm Pool.

In [12]:
res = ds.search_studies(max_lat=5, min_lat=-5, max_lon=109,
                       min_lon=125)

[2026-04-06 16:04:29,003][INFO] - search_studies: Limit defaulted to 100 (PyleoTUPS).
[2026-04-06 16:04:29,003][INFO] - search_studies: Input Query includes geographical bounds. Inspect the results to ensure they match your intended region as one study can contain sites across various parts of the world.


Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&limit=100&minLat=-5&maxLat=5&minLon=125&maxLon=109


Parsing NOAA studies: 100%|██████████| 100/100 [00:00<00:00, 1351.01it/s]
[2026-04-06 16:04:31,888][INFO] - Retrieved 100 studies.


<div style="border-left: 4px solid #d62728; background: #ffe6e6; padding: 0.5em 1em; border-radius: 4px;">
  <strong>Warning:</strong> One of the default parameters in `search_studies` is how many studies can be returned by query. The default is 100. 
</div> 

We got 100 datasets, which means we may have hit the limit. Let's increase to 500 and see what happens:

In [13]:
res = ds.search_studies(max_lat=5, min_lat=-5, max_lon=109,
                       min_lon=125, limit=500)

[2026-04-06 16:04:35,946][INFO] - search_studies: Limit set to 500.
[2026-04-06 16:04:35,947][INFO] - search_studies: Input Query includes geographical bounds. Inspect the results to ensure they match your intended region as one study can contain sites across various parts of the world.


Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&limit=500&minLat=-5&maxLat=5&minLon=125&maxLon=109


Parsing NOAA studies: 100%|██████████| 387/387 [00:00<00:00, 7506.70it/s]
[2026-04-06 16:04:42,119][INFO] - Retrieved 387 studies.


We have 387 studies now, far from the maximum of 500 so it seams we collected all records. Let's have a look at the first few rows of the results: 

In [14]:
display(res.head())

,StudyID,XMLID,StudyName,DataType,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,"Coverage [S, N, W, E]",StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding
0,11194,9632,"1,100 Year El Niño/Southern Oscillation (ENSO)...",CLIMATE RECONSTRUCTIONS,1050.0,-52.0,900.0,2002.0,"(-20.0, 20.0, 145.0, -80.0)",An index of canonical ENSO variability for the...,"[ENSO, Atmospheric and Oceanic Circulation Pat...","Jinbao Li, Shang-Ping Xie, Edward Cook, Gang H...","[{'Author': 'Li, J., S.-P. Xie, E.R. Cook, G. ...","[[{'DataTableID': '19791', 'DataTableName': 'e...",[{'fundingAgency': 'US National Science Founda...
1,22031,20009,1200 Year Atlantic Multidecadal Variability an...,CLIMATE RECONSTRUCTIONS,1150.0,-60.0,800.0,2010.0,"(0.0, 70.0, -80.0, 0.0)",Summer (May-September) Atlantic Multidecadal V...,[Atmospheric and Oceanic Circulation Patterns ...,"Jianglin Wang, Bao Yang, Fredrik Ljungqvist, J...","[{'Author': 'Jianglin Wang, Bao Yang, Fredrik ...","[[{'DataTableID': '33108', 'DataTableName': 'W...",[{'fundingAgency': 'National Natural Science F...
2,39047,80187,1500 Year Sedimentological and Geochemical Dat...,PALEOLIMNOLOGY,1200.0,-50.0,750.0,2000.0,"(-13.933, 10.712, -77.615, -65.17)",Elevations of lakes: Tota = 3015m; Siscunsi = ...,[Precipitation Reconstruction],"Broxton Bird, Byron Steinman, Jaime Escobar, A...","[{'Author': 'Bird, B.W., B.A. Steinman, J. Esc...","[[{'DataTableID': '51864', 'DataTableName': 'M...","[{'fundingAgency': 'Indiana University', 'fund..."
3,2614,1685,350 KYr Sea Level Reconstruction and Foraminif...,PALEOCEANOGRAPHY,361500.0,1000.0,-359550.0,950.0,"(2.25, 2.25, -90.95, -90.95)",,[Sea Level Reconstruction],"David Lea, Pamela Martin, Dorothy Pak, Howard ...","[{'Author': 'Lea, D.W., P.A. Martin, D.K. Pak,...","[[{'DataTableID': '4301', 'DataTableName': 'TR...",[]
4,14632,12613,700 Year El Niño/Southern Oscillation (ENSO) N...,CLIMATE RECONSTRUCTIONS,649.0,-55.0,1301.0,2005.0,"(-5.0, 5.0, -170.0, -120.0)",An index of canonical El Niño/Southern Oscilla...,"[ENSO, Atmospheric and Oceanic Circulation Pat...","Jinbao Li, Shang-Ping Xie, Edward Cook, Marian...","[{'Author': 'Li, J., S.-P. Xie, E.R. Cook, G. ...","[[{'DataTableID': '24679', 'DataTableName': 'L...",[{'fundingAgency': 'US National Science Founda...


Let's refine this search for DataType 'Paleoceanography' and 